# PolicyProof AI — Tiny Aya Colab Baseline (T4)

**Lexsi 2026 · Experimental inference only.** This notebook verifies your Hugging Face model access, loads the unchanged Tiny Aya model, and benchmarks a fictional policy example. It does **not** fine-tune a model or demonstrate improved safety.

Before running: **Runtime → Change runtime type → T4 GPU**. A read-only Hugging Face token must belong to the account granted access to `CohereLabs/tiny-aya-global`. Never place tokens in cells or GitHub.

## 1. Confirm the T4 GPU

In [ ]:
import torch, subprocess
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Connect to a Colab T4 GPU runtime.")
print("GPU:", torch.cuda.get_device_name(0))
subprocess.run(["nvidia-smi"], check=True)

## 2. Install Hugging Face libraries

In [ ]:
%pip -q install -U "transformers>=4.57.1" accelerate huggingface_hub safetensors sentencepiece

## 3. Sign in securely and verify gated access

Create a **read-only** token at https://huggingface.co/settings/tokens. Enter it only through the login popup below.

In [ ]:
from huggingface_hub import notebook_login, whoami, hf_hub_download
notebook_login()
MODEL_ID = "CohereLabs/tiny-aya-global"
hf_hub_download(repo_id=MODEL_ID, filename="config.json")
print("ACCESS CONFIRMED for", whoami()["name"])

## 4. Download PolicyProof and generate the integrity-checked dataset

In [ ]:
import subprocess, pathlib, os, sys
PROJECT = pathlib.Path("/content/PolicyProof-AI")
if not PROJECT.exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/betterlifestyle04-commits/PolicyProof-AI.git",
                    str(PROJECT)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT), "pull", "--ff-only"], check=True)
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
subprocess.run([sys.executable, "-m", "scripts.prepare"], check=True)
subprocess.run([sys.executable, "-m", "scripts.audit_dataset", "--data-dir", "data"], check=True)
CODE_REVISION = subprocess.check_output(["git","rev-parse","HEAD"], text=True).strip()
print("GitHub commit:", CODE_REVISION)

## 5. Load Tiny Aya on the T4 in float16

The published model weights are BF16; a T4 GPU does not natively support BF16 computation, so this notebook explicitly loads FP16. This is the **unchanged baseline** checkpoint.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.float16, device_map="auto",
    low_cpu_mem_usage=True, attn_implementation="sdpa"
)
model.eval()
MODEL_REVISION = getattr(model.config, "_commit_hash", None)
print("Baseline loaded:", MODEL_ID, "revision:", MODEL_REVISION)
print("GPU allocated GiB:", round(torch.cuda.memory_allocated() / 1024**3, 2))

## 6. Generate and score one real model answer

JSON errors or wrong answers are valid research findings; do not hide them. The strict score compares a single response to synthetic gold labels and is not a real-world safety measure.

In [ ]:
from policyproof.core import read_jsonl, make_prompt, score, parse_prediction
cases = read_jsonl(PROJECT / "data/unseen_policy_test.jsonl")
def generate_one(example):
    formatted = tokenizer.apply_chat_template(
        [{"role":"user","content":make_prompt(example)}],
        tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        outputs = model.generate(**formatted, max_new_tokens=160,
                                 do_sample=False, pad_token_id=tokenizer.eos_token_id)
    answer_tokens = outputs[0][formatted["input_ids"].shape[-1]:]
    return tokenizer.decode(answer_tokens, skip_special_tokens=True).strip()
example = cases[0]
prediction = generate_one(example)
print("Example:", example["id"], example["question"])
print("Real model response:", prediction)
print("Valid JSON:", parse_prediction(prediction) is not None)
print("Strict single-case score:", score(example,prediction)["correct"])

## 7. Optional full 48-example baseline

Run only after the single-question generation works. Change `RUN_FULL_BASELINE` to `True` and run this cell; it may take substantial GPU time. Reports are saved to temporary Colab storage for review, not automatically published.

In [ ]:
RUN_FULL_BASELINE = False
if RUN_FULL_BASELINE:
    import json, datetime
    from policyproof.core import aggregate, sha256
    results = []
    for i, case in enumerate(cases, 1):
        text_out = generate_one(case)
        results.append({"id":case["id"],"raw":text_out,"score":score(case,text_out)})
        if i % 8 == 0: print("Completed",i,"of",len(cases))
    report = {
        "project":"PolicyProof AI / Lexsi 2026",
        "engine":"transformers","is_real_model_result":True,
        "model":MODEL_ID,"model_revision":MODEL_REVISION,
        "model_dtype":"float16","code_commit":CODE_REVISION,
        "is_finetuned":False,"split":"unseen-policy",
        "dataset_sha256":sha256(PROJECT/"data/unseen_policy_test.jsonl"),
        "created_utc":datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "metrics":aggregate([r["score"] for r in results]),
        "scores":[r["score"] for r in results],
        "predictions":[{"id":r["id"],"raw":r["raw"]} for r in results],
        "warning":"Actual baseline inference on invented policies; not fine-tuned and not a safety-improvement claim."
    }
    destination = PROJECT/"artifacts"
    destination.mkdir(exist_ok=True)
    path = destination/"tiny_aya_base_unseen_policy.json"
    path.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding="utf-8")
    print("Saved baseline report:",path)
    print(json.dumps(report["metrics"],indent=2))
else:
    print("Full run disabled; inspect the one-question baseline first.")

## Next milestone

After this genuine baseline works, we will run CuratorKIT curation, validate AlignTune fine-tuning on suitable hardware, and use AuditKIT to compare unchanged vs. tuned checkpoints. **Do not buy compute or train blindly.** Share only cell outputs, not access tokens.